# Zero-shot classification: labels without training
Cultural Data Analysis · Week 4, code alongs 10 and 11 (optional)

**Nothing to set up:** this runs on Colab's standard runtime, no GPU needed. Run the cells one by one from the top (or Runtime → **Run all**).

On the slides we used BART (`facebook/bart-large-mnli`, about 1.6 GB). Here we use a much smaller zero-shot model, [deberta-v3-xsmall-zeroshot](https://huggingface.co/MoritzLaurer/deberta-v3-xsmall-zeroshot-v1.1-all-33) (about 140 MB), so it downloads and runs quickly. Larger models come in the workshops.

In [ ]:
# The State of the Union speeches, as in code along 1
import pandas as pd

sotu = pd.read_csv("https://raw.githubusercontent.com/goto4711/cdai/refs/heads/main/sotu_all_new.csv")
# Washington's first address, 8 January 1790
washington = sotu.loc[0, "speechtext"]
print(sotu.shape)

In [ ]:
# A small zero-shot model from Hugging Face (about 140 MB, downloaded the first time)
# A warning about HF_TOKEN may appear: you can ignore it
from transformers import pipeline

classifier = pipeline("zero-shot-classification",
                      model="MoritzLaurer/deberta-v3-xsmall-zeroshot-v1.1-all-33")

## Code along 10: the example from the slides, then Washington
The model gets a text and **your own labels**, and says how well each label fits. None of the labels was part of its training.

In [ ]:
# The example from the slides
sentence = ("The Black-Scholes model is a mathematical model for the "
            "dynamics of a financial market containing derivative "
            "investment instruments.")
labels = ["finance", "science", "business", "stock market", "research"]
result = classifier(sentence, labels)
# Labels from best to worst, with scores
print(result["labels"])
print([round(s, 2) for s in result["scores"]])

In [ ]:
# Washington's opening: the first 300 characters, with two labels for a sentiment
opening = washington[:300]
result = classifier(opening, ["positive", "negative"])
print(result["labels"])
print([round(s, 2) for s in result["scores"]])

Does the small model put *finance* first, as BART did on the slides? Does it agree with the chatbot's verdict on Washington from code along 1? Try your own labels on the opening, for example *economy, war, agriculture, education*.

## Code along 11: do we agree with the model?
Ten sentences from Biden's 2022 address, picked at random (the same ten for everyone). First you label them, then the model does, then you compare.

In [ ]:
import re

speech = sotu[sotu.year == 2022].speechtext.iloc[0]
# Split the address into sentences and keep the longer ones
sentences = re.split(r"(?<=[.!?])\s+", speech)
long_sentences = pd.Series([s for s in sentences if len(s.split()) > 15])
# Ten at random – the same ten for everyone
sample = long_sentences.sample(10, random_state=4711)
for i, s in enumerate(sample, 1):
    print(i, s[:120])

# The labels to choose from
labels = ["economy", "health", "war and security", "infrastructure", "family"]

**Before you run the next cell:** read the ten sentences and write your own label for each into `mine`, in order, choosing from the five labels above. The cell takes a minute or two: the model reads each sentence five times, once for each label.

In [ ]:
# Your label for each of the ten sentences, in order
mine = ["", "", "", "", "", "", "", "", "", ""]
if "" in mine:
    raise SystemExit("Fill in all ten labels in mine first.")

# The model's best label for each sentence – this takes a minute or two
model = [classifier(s, labels)["labels"][0] for s in sample]

# Compare
for i, (m, y) in enumerate(zip(model, mine), 1):
    print(i, "model:", m, "| you:", y, "" if m == y else " <- read again")
print(sum(m == y for m, y in zip(model, mine)), "of 10 agree")

## For class
- Read again every sentence where you disagree. Who is right – or are both labels defensible?
- Sentences 2 and 10 are about sanctions against Russia: the economy, or war and security?
- Change the labels and run the last cell again. Does the agreement change? Who decides which labels there are?